In [2]:
import pandas as pd

In [3]:
data_path = "/mnt/data1/klug/datasets/meditron/Meditron_ICU.xlsx"

In [4]:
df = pd.read_excel(data_path)

In [5]:
df.head()

,Created At,Question,Specialty,Contribution ID,Vote,User ID,First Answer Improved,Is Review,Review Count,Second Likert Count,...,Second Eval Alignment with Guidelines,Second Eval Question Comprehension,Second Eval Logical Reasoning,Second Eval Relevance & Completeness,Second Eval Harmlessness,Second Eval Fairness,Second Eval Contextual Awareness,Second Eval Your Confidence,Second Eval Model Confidence,Second Eval Communication & Clarity
0,2025-02-23T13:13:19.090Z,\nIâ€™m a specialist in intensive care medicin...,Intensive care medicine,NaN,2,NaN,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-02-26T09:44:02.379Z,\nI am an intensive care specialist in a terti...,Intensive care medicine,NaN,2,NaN,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-02-23T15:24:11.726Z,I am an intensive care specialist in a univers...,Intensive care medicine,NaN,2,NaN,NaN,NaN,3.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-02-23T13:16:54.330Z,Iâ€™m an intensive care specialist working in ...,Intensive care medicine,NaN,2,NaN,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-02-23T13:49:09.188Z,Iâ€™m a specialist in intensive care medicine ...,Intensive care medicine,NaN,2,NaN,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.columns

Index(['Created At', 'Question', 'Specialty', 'Contribution ID', 'Vote',
       'User ID', 'First Answer Improved', 'Is Review', 'Review Count',
       'Second Likert Count', 'Second Answer', 'Country', 'Name',
       'Number of Tags', 'Ideal Answer', 'First Answer', 'Tags',
       'Working Group', 'Second Answer Improved',
       'Eval Alignment with Guidelines', 'Eval Question Comprehension',
       'Eval Logical Reasoning', 'Eval Relevance & Completeness',
       'Eval Harmlessness', 'Eval Fairness', 'Eval Contextual Awareness',
       'Eval Your Confidence', 'Eval Model Confidence',
       'Eval Communication & Clarity', 'Second Eval Alignment with Guidelines',
       'Second Eval Question Comprehension', 'Second Eval Logical Reasoning',
       'Second Eval Relevance & Completeness', 'Second Eval Harmlessness',
       'Second Eval Fairness', 'Second Eval Contextual Awareness',
       'Second Eval Your Confidence', 'Second Eval Model Confidence',
       'Second Eval Communication & 

In [7]:
first_answer_df = df[['First Answer', 'Question', 'Name', 
                      'Eval Alignment with Guidelines', 'Eval Question Comprehension',
       'Eval Logical Reasoning', 'Eval Relevance & Completeness',
       'Eval Harmlessness', 'Eval Fairness', 'Eval Contextual Awareness',
       'Eval Your Confidence', 'Eval Model Confidence',
       'Eval Communication & Clarity']]
first_answer_df.rename(columns={'First Answer': 'Answer'}, inplace=True)

second_answer_df = df[['Second Answer', 'Question', 'Name', 
                       'Second Eval Alignment with Guidelines',
       'Second Eval Question Comprehension', 'Second Eval Logical Reasoning',
       'Second Eval Relevance & Completeness', 'Second Eval Harmlessness',
       'Second Eval Fairness', 'Second Eval Contextual Awareness',
       'Second Eval Your Confidence', 'Second Eval Model Confidence',
       'Second Eval Communication & Clarity']]
second_answer_df.rename(columns={'Second Answer': 'Answer', 
                                 'Second Eval Alignment with Guidelines': 'Eval Alignment with Guidelines',
       'Second Eval Question Comprehension': 'Eval Question Comprehension',
       'Second Eval Logical Reasoning': 'Eval Logical Reasoning',
       'Second Eval Relevance & Completeness': 'Eval Relevance & Completeness',
       'Second Eval Harmlessness': 'Eval Harmlessness',
       'Second Eval Fairness': 'Eval Fairness',
       'Second Eval Contextual Awareness': 'Eval Contextual Awareness',
       'Second Eval Your Confidence': 'Eval Your Confidence',
       'Second Eval Model Confidence': 'Eval Model Confidence',
       'Second Eval Communication & Clarity': 'Eval Communication & Clarity'}, inplace=True)

all_answers_df = pd.concat([first_answer_df, second_answer_df], ignore_index=True)

/tmp/ipykernel_320374/1593965511.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_answer_df.rename(columns={'First Answer': 'Answer'}, inplace=True)
/tmp/ipykernel_320374/1593965511.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  second_answer_df.rename(columns={'Second Answer': 'Answer',


In [9]:
domains = ['Eval Alignment with Guidelines', 'Eval Question Comprehension',
       'Eval Logical Reasoning', 'Eval Relevance & Completeness',
       'Eval Harmlessness', 'Eval Fairness', 'Eval Contextual Awareness',
       'Eval Your Confidence', 'Eval Model Confidence',
       'Eval Communication & Clarity']

In [10]:
import krippendorff

#  get agreement across domains: both in terms of std of answers and krippendorf alpha
answer_eval_agreement_std = pd.DataFrame(columns=['Answer', 'Question'] + domains)
answer_eval_agreement_krippendorf = pd.DataFrame(columns=['Answer', 'Question'] + domains)

for answer in all_answers_df['Answer'].unique():
    answer_df = all_answers_df[all_answers_df['Answer'] == answer]
    question = answer_df['Question'].iloc[0]
    row_std = {'Answer': answer, 'Question': question}
    row_alpha = {'Answer': answer, 'Question': question}
    for domain in domains:
        domain_std = answer_df[domain].std()
        row_std[domain] = domain_std
        # krippendorff alpha
        data = answer_df[domain].values
        # if only one non nan value, set alpha to nan
        if len(data[~pd.isna(data)]) <= 1:
            domain_alpha = float('nan')
        # elif all values are the same, set alpha to 1
        elif len(set(data[~pd.isna(data)])) == 1:
            domain_alpha = 1.0
        else:
            domain_alpha = krippendorff.alpha(reliability_data=data, level_of_measurement='ordinal', value_domain=[0, 1, 2, 3, 4, 5])
        row_alpha[domain] = domain_alpha
        
    answer_eval_agreement_std = pd.concat([answer_eval_agreement_std, pd.DataFrame([row_std])], ignore_index=True)
    answer_eval_agreement_krippendorf = pd.concat([answer_eval_agreement_krippendorf, pd.DataFrame([row_alpha])], ignore_index=True)
    

/tmp/ipykernel_320374/3786706805.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  answer_eval_agreement_std = pd.concat([answer_eval_agreement_std, pd.DataFrame([row_std])], ignore_index=True)
/tmp/ipykernel_320374/3786706805.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  answer_eval_agreement_krippendorf = pd.concat([answer_eval_agreement_krippendorf, pd.DataFrame([row_alpha])], ignore_index=True)


In [13]:
answer_eval_agreement_std['Eval Alignment with Guidelines'].describe()

count    231.000000
mean       0.723325
std        0.517691
min        0.000000
25%        0.577350
50%        0.577350
75%        1.000000
max        2.121320
Name: Eval Alignment with Guidelines, dtype: float64